# Arrow/Polars Columnar Ingestion vs LevelDB Batch Ingestion

This notebook compares the existing LevelDB object batch ingestion path with the new serialized Arrow/Polars columnar ingestion APIs. The columnar path requires caller-provided serialized `node_value` and `edge_value` payloads, so the benchmark keeps the same PyGraphDB serializer semantics while avoiding per-row backend calls when `PyRexStore` exposes native `write_columnar_batch`.

## Install Dependencies

For local development, run the notebook from a Python 3.12 kernel with the project installed in editable mode. On Colab or a clean environment, enable the install cell below.

In [ ]:
#@title Optional installs
INSTALL_DEPS = False  #@param {type: "boolean"}

if INSTALL_DEPS:
    %pip install 'pygraphdb[fast-ingest,leveldb]'

## Imports and Availability Checks

In [ ]:
from __future__ import annotations

import importlib.util
import random
import shutil
import sys
import tempfile
import time
from pathlib import Path

repo_src = Path.cwd().parent / 'src'
if repo_src.exists() and str(repo_src) not in sys.path:
    sys.path.insert(0, str(repo_src))

from pygraphdb.graphdb import Edge, GraphDB, Node
from pygraphdb.kvstores import LevelDBStore, PyRexStore
from pygraphdb.serializers import PickleSerializer

HAS_PYARROW = importlib.util.find_spec('pyarrow') is not None
HAS_POLARS = importlib.util.find_spec('polars') is not None
HAS_PYREX = importlib.util.find_spec('pyrex') is not None

print({'pyarrow': HAS_PYARROW, 'polars': HAS_POLARS, 'pyrex': HAS_PYREX})

## Benchmark Parameters

Increase `NUM_NODES` and `NUM_EDGES` for a more realistic run. The defaults are intentionally small enough for laptops and CI-like notebook environments.

In [ ]:
NUM_NODES = 10_000
NUM_EDGES = 50_000
BATCH_SIZE = 10_000
SEED = 42

EDGE_TYPES = ['drug-to-protein', 'protein-to-disease', 'drug-to-disease']

## Generate Attributed Nodes and Typed Edges

In [ ]:
def make_graph_objects(num_nodes: int, num_edges: int, seed: int):
    rng = random.Random(seed)
    nodes = [
        Node(node_id=f'n{idx}', properties={'kind': 'entity', 'group': idx % 10})
        for idx in range(num_nodes)
    ]
    edges = []
    for idx in range(num_edges):
        source = f'n{rng.randrange(num_nodes)}'
        target = f'n{rng.randrange(num_nodes)}'
        edge_type = EDGE_TYPES[idx % len(EDGE_TYPES)]
        edges.append(
            Edge(
                edge_id=f'e{idx}',
                source=source,
                target=target,
                properties={'type': edge_type, 'weight': idx % 100},
            )
        )
    return nodes, edges

nodes, edges = make_graph_objects(NUM_NODES, NUM_EDGES, SEED)
nodes[:2], edges[:2]

## Pre-Serialize Values

The columnar ingestion APIs require serialized values. This keeps attributed nodes and edges fully compatible with `GraphDB.get_node` and `GraphDB.get_edge`. The timing below reports serialization separately from backend ingestion so you can see both costs.

In [ ]:
def timed(label, func):
    start = time.perf_counter()
    result = func()
    elapsed = time.perf_counter() - start
    print(f'{label}: {elapsed:.4f}s')
    return result, elapsed

serializer_graph = GraphDB(LevelDBStore(path=tempfile.mkdtemp(prefix='pygraphdb_serializer_leveldb_')), PickleSerializer())
try:
    node_values, node_serialize_time = timed(
        'serialize node values',
        lambda: [serializer_graph.serialize_node_value(node) for node in nodes],
    )
    edge_values, edge_serialize_time = timed(
        'serialize edge values',
        lambda: [serializer_graph.serialize_edge_value(edge) for edge in edges],
    )
finally:
    serializer_graph.close()

In [ ]:
node_ids = [node.get_id for node in nodes]
edge_ids = [edge.get_id for edge in edges]
sources = [edge.source for edge in edges]
targets = [edge.target for edge in edges]
edge_types = [edge.get_type for edge in edges]

len(node_ids), len(edge_ids), len(node_values), len(edge_values)

## LevelDB Object Batch Baseline

This is the existing Python object path: create `Node`/`Edge` objects, then call `put_nodes` and chunked `put_edges_bulk(..., check_existing=False)`. It updates both edge records and adjacency indexes through Python batch APIs.

In [ ]:
def chunks(items, chunk_size):
    for start in range(0, len(items), chunk_size):
        yield items[start:start + chunk_size]

def benchmark_leveldb_object_batch():
    path = tempfile.mkdtemp(prefix='pygraphdb_leveldb_batch_')
    graph = GraphDB(LevelDBStore(path=path), PickleSerializer())
    try:
        _, node_time = timed('LevelDB put_nodes', lambda: graph.put_nodes(nodes))

        def insert_edges():
            for edge_chunk in chunks(edges, BATCH_SIZE):
                graph.put_edges_bulk(edge_chunk, check_existing=False)

        _, edge_time = timed('LevelDB put_edges_bulk append-only', insert_edges)
        assert graph.get_node(b'n0') is not None
        assert graph.get_edge(b'e0') is not None
        assert graph.neighbors_by_edge_type(sources[0], edge_types[0], direction='out')
        return {'backend': 'leveldb', 'mode': 'object-batch', 'node_time': node_time, 'edge_time': edge_time}
    finally:
        graph.close()
        shutil.rmtree(path, ignore_errors=True)

leveldb_result = benchmark_leveldb_object_batch()
leveldb_result

## Arrow Columnar Ingestion With PyRex/RocksDB

This path writes serialized node records and typed edge adjacency records through `GraphDB.ingest_nodes_arrow` and `GraphDB.ingest_edges_arrow`. With `pyrex-rocksdb>=0.3.0a0`, `PyRexStore` uses native `write_columnar_batch`.

In [ ]:
def benchmark_pyrex_arrow():
    if not HAS_PYARROW or not HAS_PYREX:
        print('Skipping Arrow/PyRex benchmark: install pyarrow and pyrex-rocksdb>=0.3.0a0')
        return None

    import pyarrow as pa

    arrow_node_ids = pa.array(node_ids, type=pa.string())
    arrow_node_values = pa.array(node_values, type=pa.binary())
    arrow_edge_ids = pa.array(edge_ids, type=pa.string())
    arrow_sources = pa.array(sources, type=pa.string())
    arrow_targets = pa.array(targets, type=pa.string())
    arrow_edge_types = pa.array(edge_types, type=pa.string())
    arrow_edge_values = pa.array(edge_values, type=pa.binary())

    path = tempfile.mkdtemp(prefix='pygraphdb_pyrex_arrow_')
    graph = GraphDB(PyRexStore(path=path, disable_wal=True), PickleSerializer())
    try:
        print('native_columnar:', graph.store.has_native_columnar_ingestion())
        _, node_time = timed(
            'PyRex ingest_nodes_arrow',
            lambda: graph.ingest_nodes_arrow(arrow_node_ids, arrow_node_values, chunk_size=BATCH_SIZE),
        )
        _, edge_time = timed(
            'PyRex ingest_edges_arrow',
            lambda: graph.ingest_edges_arrow(
                arrow_edge_ids,
                arrow_sources,
                arrow_targets,
                arrow_edge_types,
                arrow_edge_values,
                chunk_size=BATCH_SIZE,
            ),
        )
        assert graph.get_node(b'n0') is not None
        assert graph.get_edge(b'e0') is not None
        assert graph.neighbors_by_edge_type(sources[0], edge_types[0], direction='out')
        return {'backend': 'rocksdb', 'mode': 'arrow-columnar', 'node_time': node_time, 'edge_time': edge_time}
    finally:
        graph.close()
        shutil.rmtree(path, ignore_errors=True)

arrow_result = benchmark_pyrex_arrow()
arrow_result

## Polars Columnar Ingestion With PyRex/RocksDB

The Polars API is a convenience wrapper. It reads the named DataFrame columns, converts them to Arrow arrays, and then uses the same columnar ingestion path.

In [ ]:
def benchmark_pyrex_polars():
    if not HAS_POLARS or not HAS_PYREX:
        print('Skipping Polars/PyRex benchmark: install polars and pyrex-rocksdb>=0.3.0a0')
        return None

    import polars as pl

    node_df = pl.DataFrame({'node_id': node_ids, 'node_value': node_values})
    edge_df = pl.DataFrame({
        'edge_id': edge_ids,
        'source': sources,
        'target': targets,
        'edge_type': edge_types,
        'edge_value': edge_values,
    })

    path = tempfile.mkdtemp(prefix='pygraphdb_pyrex_polars_')
    graph = GraphDB(PyRexStore(path=path, disable_wal=True), PickleSerializer())
    try:
        print('native_columnar:', graph.store.has_native_columnar_ingestion())
        _, node_time = timed(
            'PyRex ingest_nodes_polars',
            lambda: graph.ingest_nodes_polars(node_df, chunk_size=BATCH_SIZE),
        )
        _, edge_time = timed(
            'PyRex ingest_edges_polars',
            lambda: graph.ingest_edges_polars(edge_df, chunk_size=BATCH_SIZE),
        )
        assert graph.get_node(b'n0') is not None
        assert graph.get_edge(b'e0') is not None
        assert graph.neighbors_by_edge_type(sources[0], edge_types[0], direction='out')
        return {'backend': 'rocksdb', 'mode': 'polars-columnar', 'node_time': node_time, 'edge_time': edge_time}
    finally:
        graph.close()
        shutil.rmtree(path, ignore_errors=True)

polars_result = benchmark_pyrex_polars()
polars_result

## Compare Results

In [ ]:
results = [result for result in [leveldb_result, arrow_result, polars_result] if result is not None]
for result in results:
    result['node_rate'] = NUM_NODES / result['node_time']
    result['edge_rate'] = NUM_EDGES / result['edge_time']

try:
    import polars as pl
    display(pl.DataFrame(results))
except Exception:
    results

## Notes

- The Arrow and Polars methods require `node_value` and `edge_value` serialized payloads. Use `graph.serialize_node_value(node)` and `graph.serialize_edge_value(edge)` when starting from Python objects.
- `ingest_edges_arrow` and `ingest_edges_polars` are append-only in this first implementation. They write edge records and typed adjacency records, but intentionally skip legacy adjacency blob rewrites.
- `PyRexStore` uses native `write_columnar_batch` when the installed `pyrex-rocksdb` exposes it. Otherwise the APIs fall back to Python bulk writes.
- Keep serialization time separate from backend ingestion time when comparing approaches. If your data already arrives with serialized binary payload columns, the columnar path avoids the object-construction cost shown above.